# Dataset - Transformando os Pares de Frases em IDs

In [8]:
from pathlib import Path
import sys

path = Path("..").resolve()

if str(path) not in sys.path:
    sys.path.append(str(path))

In [9]:
import pandas as pd
import torch
import tokenizers
from torch.utils.data import DataLoader

data = pd.read_csv(
    '../data/processed/tatoeba_data/en-pt-pairs.tsv',
    sep="\t"
)

In [10]:
tokenizer = tokenizers.Tokenizer.from_file(
    path = '../data/tokenizer/en-pt-cased-pont.json'
)


In [11]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [12]:
class Pair:
    def __init__(
        self, 
        src_ids,
        tgt_ids,
        src_attention,
        tgt_attention
    ):

        self.src_ids = src_ids
        self.tgt_ids = tgt_ids
        self.src_attention = src_attention
        self.tgt_attention = tgt_attention
        

In [13]:
'''
Primeiramente eu devo criar uma função para fazer tokenização, padding e truncation dos batches
'''

def collate_fn(batch):
    #extraindo o texto para tokenização
    train_src_txt = [row['en_sentences'] for row in batch]
    train_tgt_txt = [f'<bos> {row['pt_sentences']} <eos>' for row in batch]

    #fazendo encoding no batch completo
    src_encoding = tokenizer.encode_batch(train_src_txt)
    tgt_encoding = tokenizer.encode_batch(train_tgt_txt)

    #criando tensores dos IDs
    src_ids = torch.tensor([encoding.ids for encoding in src_encoding])
    tgt_ids = torch.tensor([encoding.ids for encoding in tgt_encoding])

    #criando tensores attention -> apenas tensores com [0] = padding (pode ser ignorado) e [1] = token válido (deve ser considerado).
    src_attention = torch.tensor([encoding.attention_mask for encoding in src_encoding])
    tgt_attention = torch.tensor([encoding.attention_mask for encoding in tgt_encoding])

    #separando inputs de outputs
    input = Pair(
        src_ids=src_ids,
        tgt_ids=tgt_ids[:, :-1], #todas as linhas, excluindo o ÚLTIMO token -> [<bos> Eu gosto de gatos] (elimina o <eos>)
        src_attention=src_attention,
        tgt_attention=tgt_attention[:, :-1]
    )

    target = tgt_ids[:, 1:] #todas as linhas, excluindo o PRIMEIRO token -> [Eu gosto de gatos <eos>] (elimina o <bos>)

    return input, target

In [14]:
def dataframe_to_dict(data):
    data = data.reset_index(drop=True)
    return data.to_dict("records")

data_len = data.shape[0]

train_data = data.iloc[:int(data_len * 0.6)]
train_dataset = dataframe_to_dict(train_data)

valid_data = data.iloc[int(data_len * 0.6) : int(data_len * 0.8)]
valid_dataset = dataframe_to_dict(valid_data)

test_data = data.iloc[int(data_len * 0.8):]
test_dataset = dataframe_to_dict(test_data)

In [15]:
#v1 do modelo
batch_size = 16
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=True
)

valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
)

In [16]:
bath = next(iter(valid_loader))

input, target = bath

[input.src_ids[0].shape for i in range(len(input.src_ids))]

input.src_ids[0].size(-1)

11

In [17]:
from src.model import Transformer

src_vocab_size = tokenizer.get_vocab_size()
tgt_vocab_size = tokenizer.get_vocab_size()

max_len = tokenizer.truncation['max_length']

#modelo reduzido para testes
model_v1 = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    max_len=max_len,
    d_ff = 1024,
    d_model = 256,
    num_heads = 4,
    num_encoder_layers = 2,
    num_decoder_layers = 2,
    dropout=0.1,
    device=device
)

In [18]:
model_v1.max_len

128

In [19]:
tokenizer.token_to_id("<bos>")

2

In [20]:
print(f"Parâmetros: {sum(p.numel() for p in model_v1.parameters()):,}")

Parâmetros: 9,838,400


# Trainemento do Modelo

In [ ]:
from src.train import train
from torch.optim import Adam
import torch.nn as nn
from torchmetrics import BLEUScore

lr = 0.0005
pad_id = tokenizer.padding['pad_id']


optimizer = Adam(
    params=model_v1.parameters(),
    betas=(0.9, 0.98),
    eps=1e-9,
    lr=lr
)

criterion = nn.CrossEntropyLoss(
    ignore_index=pad_id
)

metric = BLEUScore(
    n_gram=2
)

#TODO: schedule e warmup ainda não estão 100%
schedule = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=3
)

warmup_schedule = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimizer, 
    T_max=20, #lr maximo,
    eta_min=0.01 #lr mínimo
)

train(
    model=model_v1,
    epochs=5,
    data_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    metric=metric,
    valid_loader=valid_loader,
    schedule=schedule,
    warmup_schedule=warmup_schedule,
    pad_id=pad_id,
    device=device
)

c:\Users\Ian Guilherme Dev\Desktop\projetos-pessoais\transformer-scratch\.venv\Lib\site-packages\torchmetrics\utilities\prints.py:62: FutureWarning: Importing `BLEUScore` from `torchmetrics` was deprecated and will be removed in 2.0. Import `BLEUScore` from `torchmetrics.text` instead.
  _future_warning(
C:\Users\Ian Guilherme Dev\Desktop\projetos-pessoais\transformer-scratch\src\train.py:105: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warmup_schedule.step()
EPOCH: 1/5:   0%|          | 33/11340 [00:04<27:52,  6.76it/s, loss=6.86]


KeyboardInterrupt: 

In [ ]:
def translate(sentence, model, tokenizer, device, max_len=30):

    model.eval()

    src = torch.tensor(
        [tokenizer.encode(sentence).ids],
        dtype=torch.long,
        device=device
    )

    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")
    pad_id = tokenizer.token_to_id("<pad>")

    tgt = torch.tensor([[bos_id]], device=device)

    with torch.no_grad():
        for _ in range(max_len):

            logits = model(src=src, tgt=tgt, pad_id=pad_id, device=device)

            next_token = logits[:, -1, :].argmax(dim=-1).item()

            if next_token == eos_id:
                break

            tgt = torch.cat([
                tgt,
                torch.tensor([[next_token]], device=device)
            ], dim=1)

            print(tokenizer.decode(tgt[0, 1:].tolist()))

    return tokenizer.decode(tgt[0, 1:].tolist())

In [ ]:
translate(
    "I need to study more.",
    model_v1,
    tokenizer,
    device
)

Preciso
Preciso estudar
Preciso estudar mais
Preciso estudar mais .


'Preciso estudar mais .'

# Treinando Modelo 2

In [ ]:
#v2 do modelo
batch_size = 32
train_loader_v2 = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=True
)

valid_loader_v2 = DataLoader(
    dataset=valid_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
)

test_loader_v2 = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
)

In [ ]:
model_v2 = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    max_len=max_len,
    d_ff = 2048,
    d_model = 512,
    num_heads = 8,
    num_encoder_layers = 6,
    num_decoder_layers = 6,
    dropout=0.1
)

In [ ]:
train(
    model=model_v2,
    epochs=10,
    data_loader=train_loader_v2,
    criterion=criterion,
    optimizer=optimizer,
    metric=metric,
    valid_loader=valid_loader_v2,
    schedule=schedule,
    warmup_schedule=warmup_schedule,
    pad_id=pad_id
)